# Día 3 — El sistema de Lorenz

### Taller: Física no lineal en el aula
Congreso de Profesores de Física, Educación Secundaria

---

Cerramos el taller con el sistema que le dio nombre a todo esto. Volvemos a las
ecuaciones diferenciales, ahora en tres dimensiones, y sobre el final vamos a
encontrar escondido adentro algo muy parecido a la parábola de ayer.

El cuaderno funciona como los otros dos: *Copiar en Drive*, después *Entorno de
ejecución → Ejecutar todas*, y la primera celda se toma unos segundos preparando
las figuras. Si alguna aparece vacía o sin controles, se vuelve a ejecutar esa
celda con Shift+Enter.

---
## 0. Preparación

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D          # noqa: activa el 3D
from ipywidgets import (FloatSlider, IntSlider, ToggleButtons, HBox, VBox, Label)
from IPython.display import display

try:
    from numba import njit
except ImportError:
    def njit(f=None, **kw):
        return f if f is not None else (lambda g: g)


def activar_figuras_vivas():
    """Enciende ipympl, que es lo que hace que las figuras respondan al mouse.
    Algunas máquinas de Colab lo traen y otras no, así que si falta lo
    instalamos. Son unos segundos y una sola vez."""
    import importlib, importlib.util, subprocess, sys
    try:
        matplotlib.rcParams.validate["backend"] = lambda s: s
    except Exception:
        pass
    if importlib.util.find_spec("ipympl") is None:
        print("instalando ipympl (sólo la primera vez)...")
        for extra in (["--no-deps"], []):
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"]
                           + extra + ["ipympl"], capture_output=True, text=True)
            importlib.invalidate_caches()
            if importlib.util.find_spec("ipympl") is not None:
                break
    try:
        import ipympl
    except Exception as e:
        print("no se pudo cargar ipympl:", type(e).__name__, e)
        return False
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    for nombre in ("widget", "ipympl"):
        try:
            get_ipython().run_line_magic("matplotlib", nombre)
            if "ipympl" in matplotlib.get_backend():
                return True
        except Exception:
            pass
    try:
        matplotlib.use("module://ipympl.backend_nbagg")
        return "ipympl" in matplotlib.get_backend()
    except Exception as e:
        print(type(e).__name__, ":", e)
        return False

VIVO = activar_figuras_vivas()

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.ioff()

SIGMA, RHO, BETA = 10.0, 28.0, 8.0/3.0     # los parámetros de Lorenz (1963)


def lienzo(ancho=7.0, alto=4.2, barra=False):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    fig.canvas.header_visible = False
    fig.canvas.toolbar_visible = barra
    fig.canvas.footer_visible = barra
    return fig, ax

def mostrar(fig, *controles):
    if controles:
        display(VBox(list(controles)))
    display(fig.canvas if VIVO else fig)

if VIVO:
    # Colab baja de internet el motor que dibuja estas figuras, y recién lo
    # hace cuando aparece la primera. Le damos una de prueba y unos segundos
    # de ventaja: si no, en un "Ejecutar todas" las primeras salen mudas.
    import time
    prueba, ejes = plt.subplots(figsize=(4.8, 0.8))
    ejes.text(0.5, 0.5, "si ves este cartel, las figuras interactivas andan",
              ha="center", va="center", fontsize=9)
    ejes.axis("off")
    prueba.canvas.header_visible = False
    prueba.canvas.toolbar_visible = False
    prueba.canvas.footer_visible = False
    display(prueba.canvas)
    time.sleep(8)
    print("Todo listo. Figuras interactivas activadas:", matplotlib.get_backend())
else:
    print("ATENCIÓN: las figuras interactivas no arrancaron.")
    print("Probá 'Entorno de ejecución → Reiniciar y ejecutar todo'.")
    print("Si sigue igual, el cuaderno funciona de todos modos, pero las figuras")
    print("quedan fijas en los valores que trae cada deslizador.")

---
## 1. La historia

En 1961 Edward Lorenz era meteorólogo en el MIT y tenía sobre el escritorio una
computadora Royal McBee LGP-30: del tamaño de un mueble, con válvulas, capaz de
hacer unas sesenta multiplicaciones por segundo. Con eso corría un modelo de doce
ecuaciones que imitaba, muy groseramente, la circulación de la atmósfera, y que
escupía en una impresora una tira de números que él leía como si fueran un
pronóstico.

Un día quiso volver a mirar un tramo de una corrida anterior. Para no empezar de
cero, tomó la impresión, tipeó los valores de un instante intermedio y dejó la
máquina trabajando. Se fue a tomar un café, porque esas corridas tardaban una
hora.

Cuando volvió, el pronóstico no se parecía en nada al anterior. Lorenz pensó
primero en una válvula quemada. Pero el problema estaba en otro lado: la impresora
mostraba tres decimales y la máquina trabajaba con seis. Donde él había tipeado
0.506, la computadora tenía guardado 0.506127.

Una diferencia de una parte en mil, en un dato inventado por una impresora, había
cambiado el clima entero. Y no era un error de la máquina: era el sistema.

En 1963 publicó *Deterministic Nonperiodic Flow*, donde se hacía cargo del asunto
con un modelo mucho más chico. Tomó el modelo de convección de Barry Saltzman (una
capa de aire calentada por abajo y enfriada por arriba), se quedó con los tres
modos más importantes del movimiento y llegó a un sistema de tres ecuaciones. Los
cálculos y las figuras de ese artículo los hizo Ellen Fetter, que trabajaba con él
programando la LGP-30.

El artículo pasó casi inadvertido durante una década fuera de la meteorología. Lo
que terminó de instalar el tema fue el título de una charla suya de 1972: *¿El
aleteo de una mariposa en Brasil desata un tornado en Texas?*. Ni siquiera el
título es de él, se lo puso el organizador de la sesión, Philip Merilees, porque
Lorenz no había mandado uno a tiempo.

---

### Las ecuaciones

$$\dot{x} = \sigma(y - x)$$
$$\dot{y} = x(\rho - z) - y$$
$$\dot{z} = xy - \beta z$$

Las variables **no** son posiciones en el espacio. Cada una mide una característica
del estado de la capa de aire:

| variable | qué mide |
|---|---|
| $x$ | la intensidad del movimiento convectivo, los rollos de aire dando vueltas |
| $y$ | la diferencia de temperatura entre las corrientes que suben y las que bajan |
| $z$ | cuánto se aparta el perfil vertical de temperatura de una recta |

Los parámetros clásicos, los que usó Lorenz, son $\sigma = 10$ (una propiedad del
fluido), $\beta = 8/3$ (la geometría de la capa) y $\rho = 28$, que es el que
manda: mide cuánto se calienta la capa por abajo.

Vale la pena mirar dónde está la no linealidad. De los nueve términos, siete son
lineales. Todo lo que vamos a ver sale de dos multiplicaciones: el $xz$ de la
segunda ecuación y el $xy$ de la tercera.

In [ ]:
@njit
def paso(x, y, z, dt, sigma, rho, beta):
    "Un paso de RK4: los mismos cuatro tanteos del día 1, con tres variables."
    a1 = sigma*(y - x);     b1 = x*(rho - z) - y;     c1 = x*y - beta*z
    xa = x + dt/2*a1; ya = y + dt/2*b1; za = z + dt/2*c1
    a2 = sigma*(ya - xa);   b2 = xa*(rho - za) - ya;  c2 = xa*ya - beta*za
    xb = x + dt/2*a2; yb = y + dt/2*b2; zb = z + dt/2*c2
    a3 = sigma*(yb - xb);   b3 = xb*(rho - zb) - yb;  c3 = xb*yb - beta*zb
    xc = x + dt*a3;   yc = y + dt*b3;   zc = z + dt*c3
    a4 = sigma*(yc - xc);   b4 = xc*(rho - zc) - yc;  c4 = xc*yc - beta*zc
    return (x + dt/6*(a1 + 2*a2 + 2*a3 + a4),
            y + dt/6*(b1 + 2*b2 + 2*b3 + b4),
            z + dt/6*(c1 + 2*c2 + 2*c3 + c4))


@njit
def integrar(x0, y0, z0, dt, pasos, sigma=SIGMA, rho=RHO, beta=BETA):
    "Repite el paso y va anotando por dónde pasó."
    X = np.empty(pasos + 1); Y = np.empty(pasos + 1); Z = np.empty(pasos + 1)
    x, y, z = x0, y0, z0
    X[0], Y[0], Z[0] = x, y, z
    for i in range(pasos):
        x, y, z = paso(x, y, z, dt, sigma, rho, beta)
        X[i+1], Y[i+1], Z[i+1] = x, y, z
    return X, Y, Z

integrar(1.0, 1.0, 1.0, 0.005, 10)          # primera llamada: compila
print("listo")

<font color="#1a73e8"><b>Un aparte opcional, para los que ya se lo están
preguntando.</b></font>

Si una diferencia de $10^{-9}$ cambia el resultado, y la computadora redondea en
cada uno de los cien mil pasos que va a dar, entonces la trayectoria que vamos a
dibujar **no es** la trayectoria verdadera que sale de la condición inicial que le
pedimos. Después de un rato, no se le parece en nada. ¿Por qué le creemos al
dibujo, entonces?

La respuesta tiene nombre: **sombreado**, o *shadowing*. Para sistemas de este
tipo se puede demostrar, y en el de Lorenz se verificó numéricamente, que aunque
la trayectoria calculada se aparte de la verdadera que arranca donde nosotros
dijimos, existe **otra** condición inicial muy cercana cuya trayectoria verdadera
acompaña de cerca a la calculada, durante todo el tramo simulado. Dicho corto: lo
que dibujamos no es *la* trayectoria que pedimos, pero es *una* trayectoria
legítima del sistema.

Y eso alcanza para casi todo lo que queremos preguntar: la forma del atractor,
cuánto tiempo pasa de cada lado, el exponente que vamos a medir en la sección 4.
Lo único que no podemos hacer es usarla como pronóstico: si preguntamos dónde
está el sistema en $t = 50$, la respuesta no sirve para nada.

Es exactamente la diferencia entre el pronóstico del tiempo a diez días, que es
imposible, y el clima, que se estudia igual.

---
## 2. El atractor

Integramos y miramos la trayectoria en el espacio $(x, y, z)$. Los puntos rojos son
dos de los equilibrios del sistema, y el negro es el origen, que es el tercero.

La figura se puede **girar arrastrando con el mouse**, que es la mejor forma de
convencerse de que las dos alas no están en el mismo plano.

In [ ]:
#@title El atractor en 3D (se gira con el mouse) { display-mode: "form" }
X, Y, Z = integrar(1.0, 1.0, 1.0, 0.005, 16000)

fig = plt.figure(figsize=(7.0, 5.6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = True
fig.canvas.footer_visible = True
ax = fig.add_subplot(111, projection="3d")
ax.plot(X[::2], Y[::2], Z[::2], lw=0.35, color="#1f4e79")

c = np.sqrt(BETA*(RHO - 1))            # los dos puntos fijos C±
ax.scatter([c, -c], [c, -c], [RHO-1, RHO-1], color="#c0392b", s=45)
ax.scatter([0], [0], [0], color="black", s=35)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("El atractor de Lorenz")
mostrar(fig)

print(f"puntos fijos:  C± = (±{c:.3f}, ±{c:.3f}, {RHO-1:.0f})   y el origen")

Hay tres cosas para mirar acá, y ninguna es obvia.

La trayectoria **nunca se cierra**. Da unas vueltas de un lado, salta al otro,
vuelve, y no repite jamás la misma curva. Tampoco se cruza a sí misma: si lo
hiciera, ese punto tendría dos futuros distintos.

Los tres equilibrios son **inestables**. No hay ningún estado de reposo al que el
sistema pueda caer, y por eso no se detiene nunca.

Y sin embargo hay un atractor: todas las condiciones iniciales terminan sobre esta
misma figura. Comparen con el día 1. Allá, el péndulo con rozamiento tenía como
atractor **un punto**, y el péndulo magnético tenía **seis puntos**. Acá el atractor
no es un punto ni una curva: es un objeto con estructura, de dimensión
fraccionaria (alrededor de 2.06, o sea ni superficie ni volumen). Por eso se lo
llama **atractor extraño**.

### 2.1 Las tres sombras

En 3D cuesta ver qué pasa. Miremos las proyecciones sobre los tres planos.

In [ ]:
#@title Las tres proyecciones { display-mode: "form" }
fig, ejes = plt.subplots(1, 3, figsize=(13, 4.2))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False
for ax, (A, B, na, nb) in zip(ejes, [(X, Y, "x", "y"), (X, Z, "x", "z"),
                                     (Y, Z, "y", "z")]):
    ax.plot(A, B, lw=0.3, color="#1f4e79")
    ax.set_xlabel(na); ax.set_ylabel(nb); ax.set_title(f"plano {na}–{nb}")
fig.tight_layout()
mostrar(fig)

La proyección $x$–$z$ es la mariposa clásica, la que todo el mundo conoce.

La $x$–$y$ tiene una información distinta: las dos alas se ven casi como una recta
inclinada, porque $x$ e $y$ están muy correlacionadas. Eso se lee directamente en
la primera ecuación, $\dot x = \sigma(y - x)$, que arrastra $x$ hacia $y$ diez veces
más rápido que cualquier otra cosa que pase en el sistema.

### 2.2 Subir la temperatura

$\rho$ mide cuánto se calienta la capa de aire por abajo. Movámoslo.

In [ ]:
#@title Explorar ρ { display-mode: "form" }
s_rho  = FloatSlider(min=0.5, max=60.0, step=0.5, value=28.0, description="ρ",
                     continuous_update=True, readout_format=".1f")
s_tmax = IntSlider(min=20, max=150, step=10, value=80, description="t max",
                   continuous_update=True)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.5, 4.4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False
orbita, = a1.plot([], [], lw=0.4, color="#1f4e79")
a1.set_xlabel("x"); a1.set_ylabel("z")
serie, = a2.plot([], [], lw=0.6, color="#2c3e50")
a2.set_xlabel("t"); a2.set_ylabel("x(t)"); a2.set_title("serie temporal de x")

def dibujar(_=None):
    rho, tmax = s_rho.value, float(s_tmax.value)
    pasos = int(tmax/0.005)
    Xr, Yr, Zr = integrar(1.0, 1.0, 1.0, 0.005, pasos, rho=rho)
    orbita.set_data(Xr, Zr)
    a1.set_xlim(Xr.min() - 2, Xr.max() + 2)
    a1.set_ylim(Zr.min() - 2, Zr.max() + 2)
    a1.set_title(f"plano x–z     ρ = {rho:.1f}")
    t = np.linspace(0, tmax, pasos + 1)
    serie.set_data(t, Xr)
    a2.set_xlim(0, tmax); a2.set_ylim(Xr.min() - 2, Xr.max() + 2)
    fig.canvas.draw_idle()

for s in (s_rho, s_tmax):
    s.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_rho, s_tmax]))

**Para probar, en este orden:**

1. **$\rho = 0.5$.** Todo muere en el origen. No hay convección: el aire no se
   mueve y el calor sube por conducción nomás.
2. **$\rho = 5$.** La trayectoria termina quieta en uno de los dos puntos rojos.
   Hay convección, pero **estacionaria**: rollos de aire girando siempre igual, en
   un sentido o en el otro según cómo arranque.
3. **$\rho = 20$.** Sigue terminando en un punto fijo, pero ahora da muchas vueltas
   antes de decidirse. Es un transitorio largo.
4. **$\rho = 24.5$.** Ya casi no se decide.
5. **$\rho = 28$.** Caos. Miren la serie temporal de $x$: los saltos entre alas no
   tienen ningún patrón.
6. **$\rho = 35$ o 45.** Sigue siendo caótico, con otra forma.

Deténganse un momento en la serie temporal de $\rho = 28$. Si nos la mostraran sin
decirnos de dónde salió, diríamos que es un registro experimental con ruido. Y no
hay ni una pizca de aleatoriedad en las tres ecuaciones que la generaron. Es el
mensaje del día 1, otra vez, ahora en una señal que podría salir de un sensor.

---
## 3. El efecto mariposa, medido

Hagamos lo mismo que con el péndulo magnético: soltamos dos trayectorias casi
idénticas y miramos cuánto tardan en separarse. La diferencia es que ahora vamos a
poder medir la separación con un número.

El panel de abajo está en **escala logarítmica**: cada marca del eje vertical es un
factor 10.

In [ ]:
#@title Dos trayectorias casi iguales { display-mode: "form" }
s_exp  = IntSlider(min=-12, max=-2, value=-9, description="log₁₀(δ)",
                   continuous_update=True)
s_t3   = IntSlider(min=20, max=80, step=5, value=45, description="t max",
                   continuous_update=True)
aviso3 = Label("")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10.5, 6.6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False
lA, = a1.plot([], [], lw=0.8, color="black", label="trayectoria A")
lB, = a1.plot([], [], lw=0.8, color="#e67e22", label="trayectoria B")
a1.set_ylabel("x(t)"); a1.legend(loc="upper right", fontsize=8)
lsep, = a2.semilogy([], [], lw=1.2, color="#c0392b")
a2.axhline(30, ls="--", color="gray")
a2.text(0.5, 40, "tamaño del atractor", fontsize=8, color="gray")
a2.set_xlabel("t"); a2.set_ylabel("distancia entre A y B")
a2.set_ylim(1e-13, 300)

def dibujar(_=None):
    d0 = 10.0**s_exp.value
    tmax = float(s_t3.value)
    pasos = int(tmax/0.005)
    t = np.linspace(0, tmax, pasos + 1)
    XA, YA, ZA = integrar(1.0, 1.0, 1.0, 0.005, pasos)
    XB, YB, ZB = integrar(1.0 + d0, 1.0, 1.0, 0.005, pasos)
    lA.set_data(t, XA); lB.set_data(t, XB)
    a1.set_xlim(0, tmax); a1.set_ylim(-25, 25)
    a1.set_title(f"separación inicial δ = {d0:.0e}")
    sep = np.sqrt((XA-XB)**2 + (YA-YB)**2 + (ZA-ZB)**2)
    lsep.set_data(t, np.maximum(sep, 1e-16))
    a2.set_xlim(0, tmax)
    pasa = np.where(sep > 30)[0]
    aviso3.value = (f"las dos trayectorias se vuelven independientes en t ≈ "
                    f"{t[pasa[0]]:.1f}" if len(pasa) else
                    "todavía no se separaron del todo")
    fig.canvas.draw_idle()

for s in (s_exp, s_t3):
    s.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_exp, s_t3]), aviso3)

**Para probar:** muevan $\delta$ desde $10^{-2}$ hasta $10^{-12}$ y miren el panel
de abajo. Hay dos cosas que ver, y la segunda es la importante.

La primera: la separación crece en **línea recta** en escala logarítmica, hasta que
se topa con el tamaño del atractor y se aplana. Recta en semilog significa
crecimiento exponencial, $d(t) \approx d_0\,e^{\lambda t}$.

La segunda: achicar $\delta$ mil veces **no** da mil veces más tiempo de
predicción. Sólo corre la recta un poco hacia la derecha. Cada factor 10 de
precisión extra compra siempre el mismo ratito adicional, y ese ratito es siempre
el mismo número de segundos.

> Ésta es la razón de fondo de que el pronóstico del tiempo tenga un horizonte de
> unos diez días y de que no se arregle con más estaciones meteorológicas.
> Mejorar los datos da rendimientos logarítmicos: para duplicar el horizonte de
> predicción habría que elevar la precisión al cuadrado.

---
## 4. El exponente de Lyapunov

Esa recta del gráfico anterior tiene una pendiente, y esa pendiente es un número
que caracteriza al sistema. Se llama **exponente de Lyapunov**:

$$d(t) \approx d_0\,e^{\lambda t}
\qquad\Longrightarrow\qquad
\lambda = \lim_{t\to\infty}\frac{1}{t}\,\ln\frac{d(t)}{d_0}$$

Y sirve como criterio, no sólo como descripción:

- $\lambda > 0$ → **caos**: las diferencias se amplifican;
- $\lambda \approx 0$ → **órbita periódica**, un ciclo que se repite;
- $\lambda < 0$ → **punto fijo**: todo se frena en un estado de reposo.

Medirlo tiene una trampa que ya vieron en el gráfico de recién: si dejamos correr
las dos trayectorias, la separación se topa con el tamaño del atractor y la recta
se aplana. A partir de ahí, el promedio se arruina.

<font color="#1a73e8"><b>Cómo se esquiva eso (detalle técnico, opcional).</b></font>
El truco se llama algoritmo de Benettin y es bastante visual: en vez de dejar que
las dos trayectorias se separen libremente, cada paso se mide cuánto creció la
separación, se anota el logaritmo de ese estirón, y **se vuelve a acercar** la
segunda trayectoria a la primera, manteniendo la dirección en que se estaban
separando. Así nunca se sale del régimen exponencial: se mide el estiramiento de a
pedacitos y se van sumando. El promedio de todos esos estirones, dividido por el
tiempo, es $\lambda$.

No hace falta entenderlo para usar el número que sale.

In [ ]:
#@title El algoritmo de Benettin { display-mode: "form" }
@njit
def lyapunov(dt=0.005, tmax=1000.0, d0=1e-8, t_trans=100.0,
             sigma=SIGMA, rho=RHO, beta=BETA):
    "Exponente de Lyapunov máximo, por el algoritmo de Benettin."
    x, y, z = 1.0, 1.0, 1.0
    for _ in range(int(t_trans/dt)):        # descartamos el transitorio:
        x, y, z = paso(x, y, z, dt, sigma, rho, beta)   # medimos SOBRE el atractor

    X2, Y2, Z2 = x + d0, y, z               # la segunda trayectoria
    suma = 0.0
    n = int(tmax/dt)
    for _ in range(n):
        x, y, z = paso(x, y, z, dt, sigma, rho, beta)
        X2, Y2, Z2 = paso(X2, Y2, Z2, dt, sigma, rho, beta)

        dx, dy, dz = X2 - x, Y2 - y, Z2 - z
        d = (dx*dx + dy*dy + dz*dz)**0.5
        suma += np.log(d/d0)                # cuánto se estiró en este paso
        k = d0/d                            # y la traemos de vuelta,
        X2, Y2, Z2 = x + dx*k, y + dy*k, z + dz*k    # sin cambiar la dirección
    return suma/(n*dt)

lyapunov(tmax=1.0)                          # primera llamada: compila
print("listo")

In [ ]:
print("cómo converge con el tiempo de medición:\n")
for tmax in [100, 300, 1000, 2000]:
    print(f"   t = {tmax:5d}     λ = {lyapunov(tmax=float(tmax)):+.4f}")
print("\n   valor de referencia en la literatura:  λ ≈ 0.9056")

El número converge alrededor de **0.90**, que es el valor que aparece en la
literatura. Dos controles rápidos para convencerse de que mide lo que decimos que
mide:

In [ ]:
print("control 1 — un régimen que no es caótico (ρ = 13, termina en un punto fijo)")
print(f"   λ = {lyapunov(tmax=500.0, rho=13.0):+.4f}   → negativo, como debe ser\n")

print("control 2 — el mismo caso caótico, con otro paso de integración")
print(f"   dt = 0.005  →  λ = {lyapunov(tmax=500.0):+.4f}")
print(f"   dt = 0.002  →  λ = {lyapunov(dt=0.002, tmax=500.0):+.4f}")
print("   → el resultado no depende del paso: estamos midiendo física, no")
print("     ruido numérico")

### Qué significa 0.90 en la práctica

Si un error se multiplica por $e^{\lambda t}$, el **tiempo de duplicación** es
$\ln 2/\lambda$. Ese número dice cuánto dura la información que tenemos.

In [ ]:
lam = 0.9056
print(f"tiempo de duplicación de un error:  ln2/λ = {np.log(2)/lam:.3f}\n")
for factor in [10, 100, 1000]:
    print(f"   para que un error se multiplique por {factor:5d}: "
          f"{np.log(factor)/lam:.2f} unidades de tiempo")
print("\n   o sea que ganar un factor 10 de horizonte cuesta siempre lo mismo:")
print("   mejorar la precisión inicial por 10.")

**Ejercicio 1.** ¿Para qué valor de $\rho$ el sistema **deja** de ser caótico?

Cambien el número de la celda de abajo y busquen dónde $\lambda$ deja de ser
positivo.

*Pistas: hay una transición bastante brusca entre 23 y 24. Después de la zona
caótica, bastante más arriba de lo que uno esperaría, el sistema vuelve a ser
periódico: prueben 100, 150 y 160. Ojo que 40 y 45 **siguen** siendo caóticos.*

In [ ]:
rho_prueba = 28.0      # ←←← CAMBIAR ESTE NÚMERO

lam = lyapunov(tmax=600.0, rho=rho_prueba)
if   lam >  0.02: estado = "CAÓTICO"
elif lam < -0.02: estado = "punto fijo: todo se frena"
else:             estado = "órbita periódica: un ciclo que se repite"
print(f"ρ = {rho_prueba}   →   λ = {lam:+.4f}   →   {estado}")

---
## 5. <font color="#1a73e8">(Opcional)</font> Lorenz esconde un mapa

<font color="#1a73e8"><b>Sección opcional</b></font>, pero es la que cierra el
círculo de los tres días. Si hay tiempo, vale la pena.

Lorenz se hizo esta pregunta: *si anoto el valor **máximo** que alcanza $z$ en cada
vuelta, ¿el próximo máximo tiene algo que ver con el anterior?*

Es decir: llamemos $z_1, z_2, z_3, \ldots$ a los máximos sucesivos de $z(t)$, y
grafiquemos $z_{n+1}$ contra $z_n$. Si el sistema fuera de verdad impredecible en
el sentido de aleatorio, eso tendría que dar una nube de puntos sin forma.

Antes de ejecutar la celda, hagan su apuesta.

In [ ]:
def maximos_de_z(dt=0.002, tmax=600.0, t_trans=50.0, **kw):
    "Los máximos locales sucesivos de z(t), ya sobre el atractor."
    pasos = int((tmax + t_trans)/dt)
    _, _, Z = integrar(1.0, 1.0, 1.0, dt, pasos, **kw)
    z = Z[int(t_trans/dt):]
    # un máximo local es un punto más alto que sus dos vecinos; lo afinamos
    # pasando una parábola por los tres
    idx = np.where((z[1:-1] > z[:-2]) & (z[1:-1] > z[2:]))[0] + 1
    picos = []
    for i in idx:
        a, b, c = z[i-1], z[i], z[i+1]
        den = a - 2*b + c
        picos.append(b - 0.125*(c - a)**2/den if den != 0 else b)
    return np.array(picos)

zm = maximos_de_z()
print(f"{len(zm)} máximos encontrados, entre {zm.min():.2f} y {zm.max():.2f}")

In [ ]:
#@title El mapa de retorno, al lado del de ayer { display-mode: "form" }
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5.2))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False

a1.plot(zm[:-1], zm[1:], ".", ms=3, color="#c0392b")
lim = [zm.min() - 0.5, zm.max() + 0.5]
a1.plot(lim, lim, ls="--", lw=1, color="gray")
a1.set_xlabel("zₙ  (máximo actual)"); a1.set_ylabel("zₙ₊₁  (máximo siguiente)")
a1.set_title("El mapa de Lorenz")
a1.set_xlim(lim); a1.set_ylim(lim); a1.set_aspect("equal")

xx = np.linspace(0, 1, 300)
a2.plot(xx, 3.9*xx*(1 - xx), lw=2, color="#1f4e79")
a2.plot([0, 1], [0, 1], ls="--", lw=1, color="gray")
a2.set_xlabel("xₙ"); a2.set_ylabel("xₙ₊₁")
a2.set_title("El mapa logístico de ayer (r = 3.9)")
a2.set_aspect("equal")
fig.tight_layout()
mostrar(fig)

No es una nube: es una **curva**, con un solo pico, igual que la parábola de ayer.

Vale la pena detenerse en lo que acaba de pasar. Tenemos tres ecuaciones
diferenciales acopladas, con trayectorias que no se repiten nunca y que viven en un
objeto de dimensión fraccionaria. Y una sola pregunta bien elegida — *¿cuál es el
próximo máximo de $z$?* — lo reduce a una función de una variable, del tipo que
ayer manejábamos con una calculadora.

Ésa es la razón de haber pasado un día entero con $x(1-x)$: el mapa logístico no es
un juguete aparte, es la estructura mínima que aparece adentro de sistemas mucho
más complicados.

Midamos qué tan buena es la reducción.

In [ ]:
# ¿qué tan fina es la curva? Comparamos su grosor con su extensión.
bins = np.linspace(zm.min(), zm.max(), 40)
grosores = []
for i in range(len(bins) - 1):
    m = (zm[:-1] >= bins[i]) & (zm[:-1] < bins[i+1])
    if m.sum() > 5:
        grosores.append(zm[1:][m].std())

grosor = np.median(grosores)
rango  = zm.max() - zm.min()
print(f"grosor típico de la curva : {grosor:.3f}")
print(f"extensión total           : {rango:.3f}")
print(f"→ la curva tiene un {100*grosor/rango:.1f} % de ancho: es una curva, "
      f"no una nube\n")

# la pendiente: si |f'| > 1 en todos lados, todo error se amplifica
orden = np.argsort(zm[:-1])
a, b = zm[:-1][orden], zm[1:][orden]
pend = np.abs(np.diff(b)/np.diff(a))
pend = pend[np.isfinite(pend)]
print(f"pendiente típica |f'| = {np.median(pend):.2f}")
print("→ mayor que 1 en casi todo el dominio")

Ese último número es la explicación completa del caos de Lorenz, dicha en el
lenguaje que construimos ayer.

En el día 2 vimos que un punto fijo atrae si $|f'| < 1$ y repele si $|f'| > 1$. Acá
$|f'| > 1$ en **todos lados**: no hay ningún punto fijo estable posible, ninguna
órbita periódica que sobreviva, y cualquier diferencia entre dos condiciones
iniciales se agranda en cada vuelta.

El atractor extraño en tres dimensiones y la parábola de la calculadora son el
mismo fenómeno, mirado de dos maneras.

---
## Ejercicio 2 — <font color="#1a73e8">(Opcional)</font> reconstruir el sistema con lo que mide un osciloscopio

Este ejercicio invierte el punto de vista, y es el que conecta todo esto con los
circuitos que vamos a armar.

Cuando midamos el circuito de Chua o el tipo Duffing, no vamos a tener las
variables del modelo: vamos a tener lo que entra por los canales del osciloscopio.
Con **dos** canales y el modo XY, la pantalla nos muestra directamente una
proyección del atractor, igual que los paneles de la sección 2.1. Pero, ¿y si
tenemos **un solo** canal? ¿Se puede decir algo del sistema con una sola señal?

La respuesta es que sí, y es el **teorema de Takens**: graficando la señal contra
sí misma retrasada un tiempo $\tau$, o sea $v(t)$ contra $v(t+\tau)$, se obtiene
una figura que no es el atractor pero es equivalente a él en lo que importa
(dimensión, exponentes, estructura). Y se obtiene **sin conocer las ecuaciones**.

Abajo, los tres paneles: la señal cruda, lo que veríamos en modo XY con dos
canales, y la reconstrucción con un solo canal y un retardo.

In [ ]:
#@title Takens: reconstruir con una sola señal { display-mode: "form" }
s_tau = FloatSlider(min=0.01, max=0.50, step=0.01, value=0.10,
                    description="retardo τ", continuous_update=True,
                    readout_format=".2f")
s_var = ToggleButtons(options=[("mido x", 0), ("mido z", 1)], value=0,
                      description="canal:")

DT = 0.005
Xe, Ye, Ze = integrar(1.0, 1.0, 1.0, DT, 24000)
corte = int(20/DT)
Xe, Ye, Ze = Xe[corte:], Ye[corte:], Ze[corte:]
te = np.arange(len(Xe))*DT

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(14.5, 4.3))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False

senal, = a1.plot([], [], lw=0.5, color="#2c3e50")
a1.set_xlabel("t"); a1.set_xlim(0, 40)
a1.set_title("la señal que entra por el canal", fontsize=10)

a2.plot(Xe, Ze, lw=0.3, color="#1f4e79")
a2.set_xlabel("x"); a2.set_ylabel("z")
a2.set_title("modo XY: dos canales", fontsize=10)

recon, = a3.plot([], [], lw=0.3, color="#6c3483")
a3.set_title("un solo canal, con retardo", fontsize=10)

def dibujar(_=None):
    v = Xe if s_var.value == 0 else Ze
    nombre = "x" if s_var.value == 0 else "z"
    senal.set_data(te, v)
    a1.set_ylim(v.min() - 2, v.max() + 2)
    a1.set_ylabel(f"{nombre}(t)")
    k = max(1, int(s_tau.value/DT))
    recon.set_data(v[:-k], v[k:])
    m0, m1 = v.min() - 2, v.max() + 2
    a3.set_xlim(m0, m1); a3.set_ylim(m0, m1)
    a3.set_xlabel(f"{nombre}(t)"); a3.set_ylabel(f"{nombre}(t + τ)")
    a3.set_title(f"un solo canal, τ = {s_tau.value:.2f}", fontsize=10)
    fig.canvas.draw_idle()

s_tau.observe(dibujar, names="value")
s_var.observe(dibujar, names="value")
dibujar()
mostrar(fig, HBox([s_tau, s_var]))

**Para probar:**

1. Con $\tau$ muy chico (0.01) todo se aplasta contra la diagonal: $v(t)$ y
   $v(t+\tau)$ son casi el mismo número y no hay información nueva.
2. Con $\tau$ muy grande (0.4 o más) la figura se enreda sobre sí misma.
3. En el medio, alrededor de $\tau = 0.10$, aparece una figura con estructura
   clarísima: dos alas, dos centros, la misma forma que el panel del medio. Y eso
   salió de **una sola señal**, sin saber las ecuaciones.
4. Ahora cambien el canal a $z$ y vuelvan a barrer $\tau$. La reconstrucción ya no
   tiene dos alas: se ven superpuestas en una sola. No es un defecto del método,
   es que $z$ no distingue de qué lado está el sistema: vale lo mismo girando para
   un lado que para el otro. **Qué variable medimos importa.**

> **Para el laboratorio:** esto es lo que hace utilizable toda la teoría del taller
> con datos experimentales. Con el osciloscopio en XY sobre dos puntos del circuito
> ya se ve el atractor; con un solo canal grabado (o con la placa de sonido, o un
> Arduino) se lo reconstruye. Y una vez que tenemos la figura, se le pueden medir
> cosas: el exponente de Lyapunov, la dimensión, el mapa de retorno de los máximos.
> El punto 4 es además una buena advertencia práctica a la hora de elegir dónde
> poner la punta del osciloscopio.

---
## 6. Cierre de los tres días

In [ ]:
#@title Los tres días en una figura { display-mode: "form" }
# Un resumen visual: los tres sistemas del taller, uno al lado del otro.
fig, (p1, p2, p3) = plt.subplots(1, 3, figsize=(14.5, 4.4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False
fig.canvas.footer_visible = False
g = 9.81

# --- día 1: el espacio de fases del péndulo ---
TH, OM = np.meshgrid(np.linspace(-2*np.pi, 2*np.pi, 19), np.linspace(-8, 8, 13))
U, V = OM, -g*np.sin(TH)
n = np.hypot(U, V) + 1e-9
p1.quiver(TH, OM, U/n, V/n, color="gray", alpha=0.4, width=0.003)
for th0, om0 in [(1.0, 0.0), (2.2, 0.0), (0.0, 5.0), (0.0, 6.8), (0.0, -6.8)]:
    th, om, dt = th0, om0, 0.02
    xs, ys = [th], [om]
    for _ in range(1200):                      # el mismo RK4 del día 1
        a1, c1 = om,             -g*np.sin(th)
        a2, c2 = om + dt/2*c1,   -g*np.sin(th + dt/2*a1)
        a3, c3 = om + dt/2*c2,   -g*np.sin(th + dt/2*a2)
        a4, c4 = om + dt*c3,     -g*np.sin(th + dt*a3)
        th += dt/6*(a1 + 2*a2 + 2*a3 + a4)
        om += dt/6*(c1 + 2*c2 + 2*c3 + c4)
        xs.append(th); ys.append(om)
    p1.plot(xs, ys, lw=1.0)
th_sep = np.linspace(-2*np.pi, 2*np.pi, 600)
sep = 2*np.sqrt(g)*np.cos(th_sep/2)
p1.plot(th_sep, sep, ls=":", lw=1.1, color="#8e44ad")
p1.plot(th_sep, -sep, ls=":", lw=1.1, color="#8e44ad")
p1.plot([-2*np.pi, 0, 2*np.pi], [0, 0, 0], "o", ms=5, color="black")
p1.plot([-np.pi, np.pi], [0, 0], "x", ms=8, color="black", mew=2)
p1.set_xlim(-2*np.pi, 2*np.pi); p1.set_ylim(-8, 8)
p1.set_xlabel("θ"); p1.set_ylabel("ω")
p1.set_title("Día 1 — el espacio de fases", fontsize=11)

# --- día 2: el diagrama de bifurcación ---
nr, nx = 700, 500
r = np.linspace(2.5, 4.0, nr)
x = np.full(nr, 0.3)
for _ in range(2000):
    x = r*x*(1 - x)
cols = np.arange(nr); cae = []
for _ in range(300):
    x = r*x*(1 - x)
    fila = (x*nx).astype(np.int64)
    ok = (fila >= 0) & (fila < nx)
    cae.append(fila[ok]*nr + cols[ok])
H = np.sqrt(np.bincount(np.concatenate(cae),
                        minlength=nx*nr).reshape(nx, nr).astype(float))
p2.imshow(H, origin="lower", extent=(2.5, 4.0, 0, 1), aspect="auto",
          cmap="gray_r", interpolation="antialiased",
          vmin=0, vmax=np.quantile(H[H > 0], 0.9))
p2.set_xlabel("r"); p2.set_ylabel("x"); p2.grid(False)
p2.set_title("Día 2 — la ruta al caos", fontsize=11)

# --- día 3: el atractor ---
p3.plot(X, Z, lw=0.3, color="#1f4e79")
p3.set_xlabel("x"); p3.set_ylabel("z")
p3.set_title("Día 3 — el atractor extraño", fontsize=11)

fig.tight_layout()
mostrar(fig)

| | Día 1 | Día 2 | Día 3 |
|---|---|---|---|
| **Sistema** | péndulos | mapa logístico | Lorenz |
| **Tipo** | ecuaciones diferenciales, 2D | mapa iterado, 1D | ecuaciones diferenciales, 3D |
| **Atractor** | uno o varios puntos | ciclos y caos | atractor extraño |
| **Herramienta** | espacio de fases | diagrama de bifurcación | Lyapunov, mapa de retorno |
| **Número medido** | — | $\delta \approx 4.67$ | $\lambda \approx 0.90$ |

**Las tres ideas del taller:**

1. **La complejidad no necesita ecuaciones complicadas.** El mapa logístico es
   $x(1-x)$; Lorenz tiene dos productos. Lo que genera la riqueza no es la
   complicación algebraica sino la no linealidad más la realimentación.

2. **Determinista no es predecible.** Sin ninguna aleatoriedad en las ecuaciones, la
   predicción tiene un horizonte finito, y mejorar los datos da rendimientos
   logarítmicos.

3. **El caos es medible.** No es una etiqueta cualitativa: $\lambda$ y $\delta$ son
   números, se calculan, se comparan con experimentos, y algunos son universales.

**Y una idea para el aula:** casi todo esto se puede hacer con una calculadora, una
planilla de cálculo o un cuaderno como éste. El mapa logístico entra en una clase.
El péndulo magnético es un objeto de escritorio. El circuito de Chua se arma con
componentes de electrónica básica y se mira en un osciloscopio. La barrera de
entrada es mucho más baja de lo que el tema sugiere.

---
### Para seguir

- **Strogatz**, *Nonlinear Dynamics and Chaos*. El libro de referencia, y es muy
  legible; el capítulo de Lorenz se puede leer de corrido.
- **Gleick**, *Chaos: Making a New Science*. El relato histórico, sin matemática.
  De ahí sale buena parte de la historia del principio.
- **Lorenz (1963)**, *Deterministic Nonperiodic Flow*. El artículo original, y es
  sorprendentemente claro.
- **May (1976)**, *Simple mathematical models with very complicated dynamics*. El
  del día 2, y termina pidiendo justamente lo que estamos haciendo acá.
- **Li & Yorke (1975)**, *Period three implies chaos*. El que le puso nombre al
  campo.